# Percepção dos Brasileiros sobre Desigualdade — Simulação com LLM

**Projeto de IA** | CESOP/UNICAMP — Pesquisa 04839

Baseado em: *Simulating Public Opinion: Comparing Distributional and Individual-Level Predictions from LLMs and Random Forests* (entropy-27-00923)

---
**Perguntas simuladas**:
- **P01** — Locais de maior desigualdade no tratamento entre negros e brancos (1º lugar)
- **P06 (A–E)** — Concordância com afirmações sobre desigualdade estrutural (Likert 1–5)

**Variáveis preditoras** (10 características sociodemográficas): Sexo, Idade, Escolaridade, Raça/Cor, Religião, Renda Pessoal, Renda Familiar, Região, Condição e Porte do município.

## 0. Instalação (Colab)

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q transformers>=4.40 accelerate bitsandbytes sentencepiece pyreadstat shap plotly
    print('OK')

## 1. Imports e Configuração

In [ ]:
import os, json, re, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pyreadstat
from scipy.stats import entropy, wasserstein_distance
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import shap
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
SEED = 42; random.seed(SEED); np.random.seed(SEED)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi':130, 'font.size':10})
print('Imports OK')

## 2. Carregamento dos Dados CESOP (Pesquisa 04839)

In [ ]:
# ── Caminho do arquivo ──────────────────────────────────────────────────────
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Ou upload direto:
    # from google.colab import files
    # up = files.upload(); SAV_PATH = list(up.keys())[0]
    SAV_PATH = '/content/drive/MyDrive/04839.sav'  # ajuste o caminho no seu Drive
else:
    SAV_PATH = '../data/raw/04839/04839.sav'

df_raw, meta = pyreadstat.read_sav(SAV_PATH)
df_lbl, _    = pyreadstat.read_sav(SAV_PATH, apply_value_formats=True)

print(f'Respondentes: {len(df_raw):,} | Variaveis: {df_raw.shape[1]}')
df_raw[['SEXO','IDADE','ESCOLARIDADE','REND2','REGIAO','P1_1','P6A']].head(4)

## 3. Exploração das Variáveis Alvo (P01 e P06)

In [ ]:
# ── Distribuições Reais ─────────────────────────────────────────────────────

P01_LABELS = {
    1: 'Rua/espaços públicos',
    2: 'Trabalho',
    3: 'Escolas/universidades',
    4: 'Ambiente familiar',
    5: 'Transporte público',
    6: 'Local onde mora',
    7: 'Shoppings/comércio',
    8: 'Hospitais/postos de saúde',
    97: 'Não existe diferença',
    99: 'Não sabe/NR'
}

P06_LABELS = {
    1: 'Concorda totalmente',
    2: 'Concorda em parte',
    3: 'Nem concorda, nem discorda',
    4: 'Discorda em parte',
    5: 'Discorda totalmente',
    99: 'Não sabe/NR'
}

P06_QUESTIONS = {
    'P6A': 'Abordagem policial é baseada em cabelo, vestimenta e cor de pele',
    'P6B': 'Maior presença de negros e indígenas nas universidades é bom para a sociedade',
    'P6C': 'Representatividade de negros, mulheres e LGBTQIA+ na política reduz desigualdades',
    'P6D': 'Mercado imobiliário garante moradia digna para toda a população',
    'P6E': 'Mudanças climáticas atingem igualmente todas as pessoas independente de cor/classe'
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

# P01
p1_valid = df_raw[df_raw['P1_1'] != 99]['P1_1']
cnt = p1_valid.value_counts().sort_index()
cnt.index = [P01_LABELS.get(i, str(i)) for i in cnt.index]
axes[0].barh(cnt.index, 100*cnt/cnt.sum(), color='steelblue', alpha=0.85)
axes[0].set_xlabel('%'); axes[0].set_title('P01 — Local com mais desigualdade racial', fontsize=9)
axes[0].tick_params(axis='y', labelsize=7)

# P06
for i, (col, title) in enumerate(P06_QUESTIONS.items(), 1):
    p6v = df_raw[df_raw[col] != 99][col]
    cnt6 = p6v.value_counts().sort_index()
    cnt6.index = [P06_LABELS.get(j, str(j)) for j in cnt6.index]
    axes[i].barh(cnt6.index, 100*cnt6/cnt6.sum(),
                 color=['#2ecc71','#82e0aa','#f0b27a','#e59866','#e74c3c'][:len(cnt6)], alpha=0.85)
    axes[i].set_xlabel('%')
    axes[i].set_title(f'P06{col[-1]} — {title[:45]}...', fontsize=8)
    axes[i].tick_params(axis='y', labelsize=7)

plt.suptitle('Distribuições Reais — CESOP 04839', fontsize=12, y=1.01)
plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/distribuicoes_reais.png', bbox_inches='tight')
plt.show()

## 4. Pré-processamento

In [ ]:
# ── Variáveis ───────────────────────────────────────────────────────────────
DEMO_COLS  = ['SEXO','IDADE','ESCOLARIDADE','RACA_COR','RELIGIÃO','REND1','REND2','REGIAO','COND','PORTE']
TARGET_P01 = 'P1_1'
TARGET_P06 = ['P6A','P6B','P6C','P6D','P6E']
ALL_TARGETS = [TARGET_P01] + TARGET_P06

def group_religion(v):
    v = int(v) if not pd.isna(v) else 99
    if v == 1:          return 1
    elif 2 <= v <= 13:  return 2
    elif v == 15:       return 3
    elif v == 16:       return 4
    elif v == 20:       return 5
    else:               return 6

def group_edu(v):
    v = int(v) if not pd.isna(v) else 0
    if v <= 3:    return 1
    elif v <= 7:  return 2
    elif v <= 11: return 3
    elif v <= 14: return 4
    else:         return 5

def group_age(v):
    v = int(v) if not pd.isna(v) else 30
    if v < 25:   return 1
    elif v < 35: return 2
    elif v < 45: return 3
    elif v < 60: return 4
    else:        return 5

df = df_raw[DEMO_COLS + ALL_TARGETS].copy()
df['RELIGIÃO']     = df['RELIGIÃO'].apply(group_religion)
df['ESCOLARIDADE'] = df['ESCOLARIDADE'].apply(group_edu)
df['IDADE']        = df['IDADE'].apply(group_age)

# Renda 98 (sem rendimento) e 99 (nao respondeu) substituidos pela moda
for col in ['REND1', 'REND2']:
    df[col] = df[col].replace({98: np.nan, 99: np.nan})

for c in DEMO_COLS:
    df[c].fillna(df[c].mode()[0], inplace=True)
    df[c] = df[c].astype(int)

mask = pd.Series([True] * len(df))
for t in ALL_TARGETS:
    mask &= (df[t] != 99)
df = df[mask].reset_index(drop=True)

df[TARGET_P01] = df[TARGET_P01].replace(97, 9).astype(int)
for t in TARGET_P06:
    df[t] = df[t].astype(int)

print(f'Dataset limpo: {len(df):,} respondentes x {len(DEMO_COLS)} variaveis demograficas')
print(f'P1_1:', df['P1_1'].value_counts().sort_index().to_dict())
df.head(3)

In [ ]:
# ── Amostragem estratificada ≥ 200 respondentes (10%) ──────────────────────
N_TOTAL  = len(df)
N_SAMPLE = max(200, int(0.10 * N_TOTAL))

df_sample = (
    df.groupby(TARGET_P01, group_keys=False)
      .apply(lambda x: x.sample(frac=N_SAMPLE/N_TOTAL, random_state=SEED))
      .head(N_SAMPLE)
      .reset_index(drop=True)
)

print(f'Total disponível : {N_TOTAL:,}')
print(f'Amostra usada    : {len(df_sample):,} ({100*len(df_sample)/N_TOTAL:.1f}%)')

## 5. Configuração do LLM (HuggingFace — sem API key)

In [ ]:
import torch
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    AutoModelForCausalLM, BitsAndBytesConfig, pipeline
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# ── Escolha do modelo ───────────────────────────────────────────────────────
# Opção padrão: flan-t5-xl (~3 GB VRAM, sem autenticação)
# Alternativa melhor: mistralai/Mistral-7B-Instruct-v0.2 (~5 GB com 4-bit)
MODEL_NAME = 'google/flan-t5-xl'
# MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.2'

In [ ]:
print(f'Carregando {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

IS_T5 = 't5' in MODEL_NAME.lower()

if IS_T5:
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        device_map='auto' if DEVICE=='cuda' else None,
        torch_dtype=torch.float16 if DEVICE=='cuda' else torch.float32,
    )
    llm_pipe = pipeline(
        'text2text-generation', model=model, tokenizer=tokenizer,
        max_new_tokens=8,
        device_map='auto' if DEVICE=='cuda' else -1
    )
else:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb, device_map='auto'
    )
    llm_pipe = pipeline(
        'text-generation', model=model, tokenizer=tokenizer,
        max_new_tokens=8, return_full_text=False, device_map='auto'
    )

print('Modelo carregado!')

## 6. Engenharia de Prompts

In [ ]:
# ── Dicionários de labels legíveis para os prompts ─────────────────────────
SEXO_LBL    = {1:'homem', 2:'mulher'}
IDADE_LBL   = {1:'entre 18 e 24 anos', 2:'entre 25 e 34 anos',
                3:'entre 35 e 44 anos', 4:'entre 45 e 59 anos', 5:'60 anos ou mais'}
EDU_LBL     = {1:'sem escolaridade ou pré-escola',
                2:'ensino fundamental I incompleto (até 4ª série)',
                3:'ensino fundamental II (até 8ª série)',
                4:'ensino médio completo',
                5:'ensino superior (completo ou incompleto)'}
RACA_LBL    = {1:'branca', 2:'preta', 3:'parda', 4:'amarela', 5:'indígena'}
REL_LBL     = {1:'católica', 2:'evangélica/protestante', 3:'espírita',
                4:'afro-brasileira', 5:'sem religião/ateu', 6:'outra religião'}
REND_LBL    = {1:'mais de 20 salários mínimos', 2:'de 10 a 20 SM',
                3:'de 5 a 10 SM', 4:'de 2 a 5 SM',
                5:'de 1 a 2 SM', 6:'até 1 SM', 98:'sem renda'}
REGIAO_LBL  = {1:'Norte', 2:'Nordeste', 3:'Sudeste', 4:'Sul', 5:'Centro-Oeste'}
COND_LBL    = {1:'capital', 2:'periferia de capital', 3:'interior'}
PORTE_LBL   = {1:'município muito pequeno (até 5 mil hab.)',
                2:'município pequeno (5–10 mil hab.)',
                3:'município pequeno (10–20 mil hab.)',
                4:'município médio (20–50 mil hab.)',
                5:'município médio (50–100 mil hab.)',
                6:'município grande (100–500 mil hab.)',
                7:'metrópole (mais de 500 mil hab.)'}

DEMO_MAP = {
    'SEXO': SEXO_LBL, 'IDADE': IDADE_LBL, 'ESCOLARIDADE': EDU_LBL,
    'RACA_COR': RACA_LBL, 'RELIGIÃO': REL_LBL,
    'REND1': REND_LBL, 'REND2': REND_LBL,
    'REGIAO': REGIAO_LBL, 'COND': COND_LBL, 'PORTE': PORTE_LBL
}


def describe_respondent(row: pd.Series) -> str:
    """Constrói descrição textual do perfil sociodemográfico."""
    parts = []
    for col in DEMO_COLS:
        v = int(row[col])
        lbl = DEMO_MAP.get(col, {}).get(v, str(v))
        if col == 'SEXO':
            parts.append(lbl)                             # 'homem'
        elif col == 'IDADE':
            parts.append(lbl)                             # 'entre 25 e 34 anos'
        elif col == 'ESCOLARIDADE':
            parts.append(f'escolaridade: {lbl}')
        elif col == 'RACA_COR':
            parts.append(f'raça/cor: {lbl}')
        elif col == 'RELIGIÃO':
            parts.append(f'religião: {lbl}')
        elif col == 'REND2':
            parts.append(f'renda familiar: {lbl}')
        elif col == 'REND1':
            pass   # evita redundância (usamos REND2)
        elif col == 'REGIAO':
            parts.append(f'região: {lbl}')
        elif col == 'COND':
            parts.append(f'mora em {lbl}')
        elif col == 'PORTE':
            parts.append(f'{lbl}')
    return ', '.join(p for p in parts if p)


# ── Templates de perguntas ─────────────────────────────────────────────────
P01_QUESTION = (
    "Pensando no acesso e atendimento dos serviços da sua cidade, "
    "em qual local você acredita que existe MAIS diferença no tratamento "
    "de pessoas negras e pessoas brancas? (escolha o principal)\n"
    "Opções: (1) Rua/espaços públicos  (2) Trabalho  (3) Escolas/universidades  "
    "(4) Ambiente familiar  (5) Transporte público  (6) Local onde mora  "
    "(7) Shoppings/comércio  (8) Hospitais/postos de saúde  "
    "(9) Não existe diferença"
)

P06_QUESTIONS_FULL = {
    'P6A': 'A abordagem policial é baseada no tipo de cabelo, vestimenta e cor de pele das pessoas.',
    'P6B': 'A maior presença de pessoas negras e indígenas nas universidades é bom para toda a sociedade.',
    'P6C': 'Aumentar a representatividade de negros, mulheres e LGBTQIA+ na política contribui para diminuir as desigualdades estruturais.',
    'P6D': 'No Brasil, a construção de moradias pelo mercado imobiliário garante acesso à moradia digna para toda a população.',
    'P6E': 'As mudanças climáticas atingem igualmente todas as pessoas, independente de cor ou classe social.'
}
LIKERT_OPTIONS = (
    "(1) Concorda totalmente  (2) Concorda em parte  "
    "(3) Nem concorda nem discorda  (4) Discorda em parte  (5) Discorda totalmente"
)


def build_prompt(row: pd.Series, target: str) -> str:
    desc = describe_respondent(row)
    base = (
        f"Você é um cidadão brasileiro com o seguinte perfil sociodemográfico:\n"
        f"{desc}.\n\n"
    )
    if target == 'P1_1':
        return base + P01_QUESTION + "\n\nResponda APENAS com o número da opção escolhida:"
    else:
        q = P06_QUESTIONS_FULL[target]
        return (
            base +
            f"Afirmação: \"{q}\"\n"
            f"Você concorda ou discorda dessa afirmação?\n"
            f"Opções: {LIKERT_OPTIONS}\n\n"
            f"Responda APENAS com o número da opção (1–5):"
        )


# Teste
print('=== Exemplo de prompt P01 ===')
print(build_prompt(df_sample.iloc[0], 'P1_1'))
print('\n=== Exemplo de prompt P6A ===')
print(build_prompt(df_sample.iloc[0], 'P6A'))

## 7. Simulação com o LLM (3 repetições)

In [ ]:
def parse_response(text: str, valid_codes: list) -> int:
    """Extrai o número da resposta do texto gerado pelo LLM."""
    text = text.strip()
    # Busca dígitos no início ou após ':'
    for pattern in [r'^(\d+)', r':(\s*)(\d+)', r'\b(\d+)\b']:
        m = re.search(pattern, text)
        if m:
            n = int(m.group().strip().lstrip(':').strip())
            if n in valid_codes:
                return n
    # fallback: amostra aleatória ponderada pela distribuição real
    return random.choice(valid_codes)


N_REPEATS  = 3   # aumentar para 5 se tiver tempo
BATCH_SIZE = 16  # ajustar conforme VRAM

VALID_CODES = {
    'P1_1': list(range(1, 10)),  # 1-8 + 9 (não existe diferença)
    **{f'P6{x}': list(range(1, 6)) for x in 'ABCDE'}
}

sim_results = {t: [] for t in ALL_TARGETS}

for run in range(N_REPEATS):
    print(f'\n--- Run {run+1}/{N_REPEATS} ---')
    for target in ALL_TARGETS:
        prompts = [build_prompt(row, target) for _, row in df_sample.iterrows()]
        preds = []
        for i in tqdm(range(0, len(prompts), BATCH_SIZE),
                      desc=f'  {target}', leave=False):
            batch = prompts[i:i+BATCH_SIZE]
            outs  = llm_pipe(batch)
            for out in outs:
                txt = out[0]['generated_text'] if isinstance(out, list) else out['generated_text']
                preds.append(parse_response(txt, VALID_CODES[target]))
        sim_results[target].append(preds)
        print(f'  {target}: distribuição simulada → {pd.Series(preds).value_counts().sort_index().to_dict()}')

print('\nSimulação concluída!')

## 8. Avaliação — Acurácia e Distribuições

In [ ]:
# ── Métricas por target e por run ──────────────────────────────────────────
records = []
for target in ALL_TARGETS:
    y_true = df_sample[target].values
    for run, preds in enumerate(sim_results[target]):
        y_pred = np.array(preds)
        valid_codes = VALID_CODES[target]

        acc = accuracy_score(y_true, y_pred)
        f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)

        # Distribuições suavizadas (Laplace)
        p_real = np.array([(y_true == c).mean() + 1e-9 for c in valid_codes])
        p_sim  = np.array([(y_pred == c).mean() + 1e-9 for c in valid_codes])
        p_real /= p_real.sum()
        p_sim  /= p_sim.sum()

        # KL Divergence
        kl = float(entropy(p_real, p_sim))

        # Earth Mover's Distance (Wasserstein)
        emd = float(wasserstein_distance(valid_codes, valid_codes, p_real, p_sim))

        # Jensen-Shannon Distance — métrica principal de Miranda & Balbi (2025)
        m   = 0.5 * (p_real + p_sim)
        jsd = float(np.sqrt(0.5 * entropy(p_real, m) + 0.5 * entropy(p_sim, m)))

        records.append(dict(target=target, run=run, acc=acc, f1=f1, kl=kl, emd=emd, jsd=jsd))

metrics = pd.DataFrame(records)
summary = (
    metrics.groupby('target')[['acc','f1','jsd','kl','emd']]
           .agg(['mean','std'])
           .round(4)
)
summary.columns = [
    'acc_mean','acc_std','f1_mean','f1_std',
    'jsd_mean','jsd_std','kl_mean','kl_std','emd_mean','emd_std'
]
print('=== Métricas do LLM (média ± dp sobre runs) ===')
print('JSD = Jensen-Shannon Distance (↓ melhor) — Miranda & Balbi, 2025')
print(summary.to_string())

## 9. Visualizações Comparativas

In [ ]:
# ── Real vs. Simulado por variável ─────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
axes = axes.flatten()

for ax, target in zip(axes, ALL_TARGETS):
    valid_codes = VALID_CODES[target]
    y_true = df_sample[target].values

    p_real = np.array([(y_true == c).mean() for c in valid_codes]) * 100

    sim_runs = np.array([[( np.array(run)==c).mean() for c in valid_codes]
                          for run in sim_results[target]]) * 100
    p_sim_mean = sim_runs.mean(axis=0)
    p_sim_std  = sim_runs.std(axis=0)

    # Labels do eixo x
    if target == 'P1_1':
        xlbls = [P01_LABELS.get(c, str(c))[:15] for c in valid_codes]
    else:
        xlbls = [P06_LABELS.get(c, str(c))[:18] for c in valid_codes]

    x = np.arange(len(valid_codes))
    w = 0.35
    ax.bar(x-w/2, p_real, w, label='Real', color='steelblue', alpha=0.88)
    ax.bar(x+w/2, p_sim_mean, w, yerr=p_sim_std,
           label=f'LLM ({N_REPEATS} runs)', color='tomato', alpha=0.88, capsize=4)
    ax.set_xticks(x)
    ax.set_xticklabels(xlbls, rotation=30, ha='right', fontsize=7)
    ax.set_ylabel('%')
    if target == 'P1_1':
        ax.set_title('P01 — Local de desigualdade racial', fontsize=9)
    else:
        ax.set_title(f'{target} — {P06_QUESTIONS[target][:40]}...', fontsize=8)
    ax.legend(fontsize=7)

plt.suptitle('Distribuições Reais vs. LLM Simulado — CESOP 04839', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../results/figures/distribuicoes_real_vs_llm.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, (col, ylabel, title) in zip(axes, [
    ('acc',  'Acurácia',            'Acurácia Individual'),
    ('f1',   'F1-Macro',            'F1-Macro'),
    ('jsd',  'JSD (↓ melhor)',      'Jensen-Shannon Distance'),
    ('kl',   'KL Divergence (↓)',   'KL Divergence'),
]):
    grp = metrics.groupby('target')[col].agg(['mean','std']).reset_index()
    grp.sort_values('mean', ascending=(col in ('jsd','kl')), inplace=True)
    color = '#e74c3c' if col in ('jsd','kl') else '#2ecc71'
    ax.barh(grp['target'], grp['mean'], xerr=grp['std'], capsize=5,
            color=color, alpha=0.85)
    ax.axvline(grp['mean'].mean(), ls='--', color='navy', alpha=0.5, label='Média')
    ax.set_xlabel(ylabel); ax.set_title(title, fontsize=9); ax.legend(fontsize=7)

plt.suptitle('Desempenho do LLM por Variável-Resposta — CESOP 04839', fontsize=12)
plt.tight_layout()
plt.savefig('../results/figures/metricas_llm.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Matrizes de confusão (run 0) ───────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for ax, target in zip(axes, ALL_TARGETS):
    y_true = df_sample[target].values
    y_pred = np.array(sim_results[target][0])
    codes  = VALID_CODES[target]
    cm     = confusion_matrix(y_true, y_pred, labels=codes)
    cm_pct = 100 * cm / (cm.sum(axis=1, keepdims=True) + 1e-9)

    if target == 'P1_1':
        lbls = [P01_LABELS.get(c,'')[:12] for c in codes]
    else:
        lbls = [P06_LABELS.get(c,'')[:14] for c in codes]

    sns.heatmap(cm_pct, ax=ax, annot=True, fmt='.0f', cmap='Blues',
                xticklabels=lbls, yticklabels=lbls, cbar=False,
                annot_kws={'size':7})
    ax.tick_params(axis='both', labelsize=6)
    ax.set_xlabel('LLM predito', fontsize=8)
    ax.set_ylabel('Real', fontsize=8)
    title = 'P01' if target=='P1_1' else target
    ax.set_title(f'Confusão — {title} (%)', fontsize=9)

plt.suptitle('Matrizes de Confusão (run 1)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../results/figures/confusion_matrices.png', bbox_inches='tight')
plt.show()

## 10. Explicabilidade — SHAP

Treinamos um Random Forest nas predições do LLM e calculamos SHAP values para identificar quais variáveis demográficas mais determinam as respostas simuladas.

In [ ]:
shap_data = {}
X_demo = df_sample[DEMO_COLS].values

for target in ALL_TARGETS:
    # Usa média dos runs como predição agregada
    y_llm = pd.DataFrame(sim_results[target]).T.median(axis=1).round().astype(int).values

    if len(np.unique(y_llm)) < 2:
        continue

    rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=SEED, n_jobs=-1)
    rf.fit(X_demo, y_llm)

    explainer   = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_demo)

    if isinstance(shap_values, list):
        mean_abs = np.abs(np.array(shap_values)).mean(axis=0).mean(axis=0)
    else:
        mean_abs = np.abs(shap_values).mean(axis=0)

    imp_df = pd.DataFrame({'feature': DEMO_COLS, 'shap': mean_abs})\
               .sort_values('shap', ascending=False)
    shap_data[target] = (imp_df, shap_values, X_demo, rf)
    print(f'{target}: top features → {imp_df.head(3)["feature"].tolist()}')

print('SHAP calculado.')

In [ ]:
n = len(shap_data)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for ax, (target, (imp_df, _, _, _)) in zip(axes, shap_data.items()):
    top = imp_df.head(10)
    ax.barh(top['feature'][::-1], top['shap'][::-1],
            color='mediumslateblue', alpha=0.85)
    ax.set_xlabel('Importância SHAP (|valor| médio)')
    title = 'P01' if target=='P1_1' else target
    ax.set_title(f'SHAP — {title}', fontsize=9)

for ax in axes[len(shap_data):]:
    ax.set_visible(False)

plt.suptitle('Importância das Variáveis Demográficas (via SHAP)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../results/figures/shap_importance.png', bbox_inches='tight')
plt.show()

## 11. (Extra) Comparação: LLM vs. Random Forest com CV 5-fold

In [ ]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
rf_records = []

for target in ALL_TARGETS:
    X = df_sample[DEMO_COLS].values
    y = df_sample[target].values

    if len(np.unique(y)) < 2:
        continue

    rf   = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
    ypred = cross_val_predict(rf, X, y, cv=skf)

    rf_records.append(dict(
        target  = target,
        rf_acc  = accuracy_score(y, ypred),
        rf_f1   = f1_score(y, ypred, average='macro', zero_division=0),
    ))

rf_df = pd.DataFrame(rf_records)

# Merge com métricas do LLM (médias dos runs)
llm_agg = metrics.groupby('target')[['acc','f1']].mean().reset_index()\
                 .rename(columns={'acc':'llm_acc','f1':'llm_f1'})
comp_df = llm_agg.merge(rf_df, on='target')
print(comp_df.round(3).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(len(comp_df))
w = 0.35

for ax, (lcol, rcol, title) in zip(axes, [
    ('llm_acc','rf_acc','Acurácia'),
    ('llm_f1', 'rf_f1', 'F1-Macro'),
]):
    ax.bar(x-w/2, comp_df[lcol], w, label='LLM',           color='tomato',    alpha=0.85)
    ax.bar(x+w/2, comp_df[rcol], w, label='Random Forest', color='steelblue', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(comp_df['target'], rotation=20, ha='right')
    ax.set_ylabel(title)
    ax.set_title(f'{title}: LLM vs. Random Forest')
    ax.legend()

plt.suptitle('Comparação LLM vs. Random Forest (CV 5-fold) — CESOP 04839', fontsize=12)
plt.tight_layout()
plt.savefig('../results/figures/llm_vs_rf.png', bbox_inches='tight')
plt.show()

## 12. Salvamento de Resultados

In [ ]:
os.makedirs('../results/metrics', exist_ok=True)

metrics.to_csv('../results/metrics/llm_metrics.csv', index=False)
comp_df.to_csv('../results/metrics/llm_vs_rf.csv', index=False)

summary_json = {
    'modelo':          MODEL_NAME,
    'artigo_base':     'Miranda & Balbi (2025) — Entropy 27, 923',
    'dataset':         'CESOP 04839 — Percepção dos Brasileiros sobre Desigualdade',
    'n_respondentes':  len(df_sample),
    'n_repeticoes':    N_REPEATS,
    'n_folds_cv':      N_FOLDS,
    'variaveis_demo':  DEMO_COLS,
    'targets':         ALL_TARGETS,
    'llm_acc_media':   round(float(metrics['acc'].mean()), 4),
    'llm_f1_media':    round(float(metrics['f1'].mean()), 4),
    'jsd_media':       round(float(metrics['jsd'].mean()), 4),
    'kl_media':        round(float(metrics['kl'].mean()), 4),
    'emd_media':       round(float(metrics['emd'].mean()), 4),
    'rf_acc_media':    round(float(rf_df['rf_acc'].mean()), 4) if len(rf_df) > 0 else None,
}
with open('../results/metrics/summary.json', 'w') as f:
    json.dump(summary_json, f, indent=2, ensure_ascii=False)

print('Resultados salvos.')
print(json.dumps(summary_json, indent=2, ensure_ascii=False))

## 13. Conclusões

*(Preencher após execução com os resultados reais)*

### Principais achados
| Métrica | LLM | Random Forest |
|---------|-----|---------------|
| Acurácia média | — | — |
| F1-macro médio | — | — |
| KL divergence  | — | — |

### Variáveis mais relevantes (SHAP)
*(a preencher)*

### Limitações
- `flan-t5-xl` foi treinado primariamente em inglês; prompts em português podem ter viés
- Respostas numéricas simples não capturam toda a nuance da escala Likert
- Amostra de 10% pode não ser representativa de grupos minoritários

### Trabalhos futuros
- Testar modelos nativos de português (BERTimbau, Sabiá)
- Aumentar para 5 repetições e 20–30% dos dados
- Explorar chain-of-thought nos prompts